# SuperCell Builder — Arbitrary Zone-Axis Supercells for STEM Simulations

**Acknowledgements:**

shamail.ahmed@physik.uni-marburg.de

*Import the core libraries: abTEM, ASE, py4DSTEM, NumPy, and matplotlib.\
*Define the file paths to locate the input CIF file.

In [ ]:
import abtem
import ase
import ase.io
import matplotlib.pyplot as plt
import numpy as np
from itertools import product

# Load the .cif file

In [2]:
cif_path = r"Z:\Supercell\LiNiO2"

cif_file_name = '\\LiNiO2.cif'

cif_file_path = cif_path + cif_file_name

# Repetition Estimation

## Transformation matrix

*Manually enter a 3×3 rotation/transformation matrix that specifies the desired crystallographic orientation of the supercell (e.g., a zone-axis tilt).\
*This matrix is parsed from a text block into a NumPy array for use in later steps.

In [ ]:
# Transformation matrix

text = """
 +0.946167  +0.106928  -0.305506
 -0.151387  +0.980451  -0.125693
 +0.286093  +0.165176  +0.943858
"""

matrix = np.array([list(map(float, line.split())) for line in text.strip().splitlines()])

print(matrix)

## Repetition Estimation

*Load the unit cell from the CIF file using ASE and read its lattice vectors.\
*Define a target cuboid size (e.g., 100 × 100 × 100 Å).\
*Simulate the bounding box that results from tiling the unit cell by (na, nb, nc) repetitions and then applying the transformation matrix — this mimics exactly what the carving step will produce.\
*Compute a sensitivity matrix: how much each additional repeat along a, b, or c grows the bounding box along each Cartesian axis.\
*Use that sensitivity to make an initial estimate of the required repetitions via a direct ceiling-division.\
*Refine the estimate with a greedy loop that iteratively identifies which Cartesian dimension is still undersized and adds repetitions until all three dimensions meet or exceed the target.\
*Print the final required repetitions (rep_a, rep_b, rep_c) and how much the predicted box overshoots the target.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1.  INPUTS
# ──────────────────────────────────────────────────────────────────────────────
unit_cell_atoms = ase.io.read(cif_file_path)

A = matrix


target_Lx = 50.0
target_Ly =50.0
target_Lz = 1500.0


target = np.array([target_Lx, target_Ly, target_Lz])

print(f"Target cuboid: {target_Lx:.2f} x {target_Ly:.2f} x {target_Lz:.2f} Å")

orig_cell = unit_cell_atoms.get_cell().array
print(f"\nOriginal unit cell vectors (Å):\n{orig_cell.round(6)}")

# ──────────────────────────────────────────────────────────────────────────────
# 2.  Ground-truth bbox function — runs the EXACT same pipeline as the
#     working carving cell:
#       (1) repeat original cell na x nb x nc
#       (2) apply affine transform A to cell vectors AND positions
#       (3) compute 8-corner bbox on the TRANSFORMED cell vectors
#     This is exactly what bbox_min/bbox_max produce in the carving cell.
# ──────────────────────────────────────────────────────────────────────────────
def simulate_bbox(na, nb, nc):
    """
    Simulate the full pipeline for (na, nb, nc) repeats and return
    the carved cuboid dimensions (Lx, Ly, Lz) — identical to what
    the carving cell produces.
    """
    # Step 1: tiled cell vectors (just scale, no atoms needed for bbox)
    tiled_cell = np.array([na * orig_cell[0],
                            nb * orig_cell[1],
                            nc * orig_cell[2]])

    # Step 2: apply affine transform to cell vectors
    trans_cell = tiled_cell @ A.T      # same as carving cell: old_cell @ A.T

    # Step 3: 8-corner bbox of the transformed parallelepiped
    a_vec, b_vec, c_vec = trans_cell[0], trans_cell[1], trans_cell[2]
    corners = np.array([
        i*a_vec + j*b_vec + k*c_vec
        for i, j, k in product([0, 1], repeat=3)
    ])
    bbox = corners.max(axis=0) - corners.min(axis=0)
    return bbox   # (Lx, Ly, Lz) in the same frame the carving cell uses

# Sanity check with 1x1x1
bbox_1 = simulate_bbox(1, 1, 1)
print(f"\nSimulated bbox of 1x1x1 (Å):")
print(f"  Lx={bbox_1[0]:.6f}  Ly={bbox_1[1]:.6f}  Lz={bbox_1[2]:.6f}")

# ──────────────────────────────────────────────────────────────────────────────
# 3.  Sensitivity: how much does +1 rep in each direction change each bbox axis?
# ──────────────────────────────────────────────────────────────────────────────
def sensitivity(na, nb, nc):
    b0 = simulate_bbox(na, nb, nc)
    da = simulate_bbox(na+1, nb,   nc  ) - b0
    db = simulate_bbox(na,   nb+1, nc  ) - b0
    dc = simulate_bbox(na,   nb,   nc+1) - b0
    return np.array([da, db, dc])   # shape (3,3)

# ──────────────────────────────────────────────────────────────────────────────
# 4.  Initial estimate: for each Cartesian axis find which rep controls it most
#     and do a direct ceil-divide
# ──────────────────────────────────────────────────────────────────────────────
sens_1 = sensitivity(1, 1, 1)
print(f"\nSensitivity at 1x1x1 (rows=rep a/b/c, cols=Lx/Ly/Lz):")
print(sens_1.round(6))

reps = np.ones(3, dtype=int)
for cart_axis in range(3):
    col = sens_1[:, cart_axis]
    dominant_rep = np.argmax(np.abs(col))
    if np.abs(col[dominant_rep]) > 1e-8:
        reps[dominant_rep] = max(reps[dominant_rep],
                                 int(np.ceil(target[cart_axis] / col[dominant_rep])))

print(f"\nInitial estimate: rep_a={reps[0]}, rep_b={reps[1]}, rep_c={reps[2]}")
print(f"Simulated bbox at initial estimate: {simulate_bbox(*reps).round(4)}")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  Greedy refinement loop
# ──────────────────────────────────────────────────────────────────────────────
for iteration in range(10000):
    bbox = simulate_bbox(*reps)
    shortfall = target - bbox

    if np.all(shortfall <= 1e-6):
        break

    worst_axis  = np.argmax(shortfall)
    sens        = sensitivity(*reps)
    axis_sens   = sens[:, worst_axis]   # how much each rep helps on worst axis

    if np.max(axis_sens) < 1e-8:
        reps += 1
        continue

    best_rep     = np.argmax(axis_sens)
    steps_needed = int(np.ceil(shortfall[worst_axis] / axis_sens[best_rep]))
    reps[best_rep] += max(1, steps_needed)

rep_a, rep_b, rep_c = reps
final_bbox = simulate_bbox(rep_a, rep_b, rep_c)

print(f"\n{'='*60}")
print(f"Required repetitions of original CIF cell:")
print(f"  rep_a = {rep_a}  (along [100])")
print(f"  rep_b = {rep_b}  (along [010])")
print(f"  rep_c = {rep_c}  (along [001])")
print(f"\nPredicted carved cuboid (Å):")
print(f"  Lx = {final_bbox[0]:.4f}  (target: {target_Lx:.4f})")
print(f"  Ly = {final_bbox[1]:.4f}  (target: {target_Ly:.4f})")
print(f"  Lz = {final_bbox[2]:.4f}  (target: {target_Lz:.4f})")
print(f"\nDifference from target (Å):")
print(f"  ΔLx = {final_bbox[0]-target_Lx:+.4f}")
print(f"  ΔLy = {final_bbox[1]-target_Ly:+.4f}")
print(f"  ΔLz = {final_bbox[2]-target_Lz:+.4f}")
print(f"{'='*60}")

# Repeat the structure in x, y and z directions

*Load the unit cell from the CIF again with ASE.\
*Tile it by the computed rep_a × rep_b × rep_c repetitions along the crystal's own lattice vectors using ASE's * operator.\
*Print the resulting atom count and cell dimensions, then save the supercell as an extended XYZ file (which preserves cell information for use in abTEM).

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1.  Load the CIF
# ──────────────────────────────────────────────────────────────────────────────
atoms = ase.io.read(cif_file_path)   # reads the primitive/conventional cell

print("Unit cell loaded:")
print(f"  Formula     : {atoms.get_chemical_formula()}")
print(f"  Lattice (Å) :\n{atoms.get_cell()}")
print(f"  # atoms     : {len(atoms)}")

# ──────────────────────────────────────────────────────────────────────────────
# 2.  Define the repetition factors along [100], [010], [001]
#     These are repetitions along the crystal's own lattice vectors a, b, c —
#     NOT Cartesian x, y, z — matching exactly what Atomsk does.
# ──────────────────────────────────────────────────────────────────────────────
rep_a = rep_a   # along [100] (lattice vector a)
rep_b = rep_b   # along [010] (lattice vector b)
rep_c = rep_c   # along [001] (lattice vector c)

# ──────────────────────────────────────────────────────────────────────────────
# 3.  Build the supercell
#     ase.build.make_supercell or the * operator both tile along lattice vectors.
#     The * operator is the cleanest for orthogonal repeats.
# ──────────────────────────────────────────────────────────────────────────────
supercell = atoms * (rep_a, rep_b, rep_c)

print(f"\nSupercell built:")
print(f"  # atoms     : {len(supercell)}")
print(f"  Cell (Å)    :\n{supercell.get_cell()}")

# ──────────────────────────────────────────────────────────────────────────────
# 4.  Write to XYZ  (extended XYZ keeps cell info — ideal for abTEM)
# ──────────────────────────────────────────────────────────────────────────────
#out_path = cif_path + cif_file_name.replace('.cif', f'_{rep_a}x{rep_b}x{rep_c}.xyz')

#ase.io.write(out_path, supercell, format='extxyz')   # extended XYZ preserves the cell
#print(f"\nSupercell written to:\n  {out_path}")

# Apply Transformation Matrix

*Take the supercell's cell vectors (stored as row vectors in ASE) and multiply by A.T to rotate them into the desired orientation.\
*Apply the same rotation to every atom's Cartesian position using the same matrix multiplication.\
*Assemble a new ASE Atoms object with the rotated cell and positions, and save it as a second XYZ file (_transformed.xyz).

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2.  Transform the cell vectors
#     ASE stores cell as row vectors: cell[i] = lattice vector i (in Cartesian Å)
#     Ovito applies A to column vectors, so: new_cell_row = (A @ old_cell_row.T).T
#     which is simply:  new_cell = old_cell @ A.T
# ──────────────────────────────────────────────────────────────────────────────
old_cell = supercell.get_cell()          # shape (3, 3), row vectors
new_cell = old_cell @ A.T                # transform each lattice vector

# ──────────────────────────────────────────────────────────────────────────────
# 3.  Transform atomic positions  (Cartesian)
# ──────────────────────────────────────────────────────────────────────────────
old_pos = supercell.get_positions()      # shape (N, 3)
new_pos = old_pos @ A.T                  # apply same rotation to every atom

# ──────────────────────────────────────────────────────────────────────────────
# 4.  Build the transformed ASE Atoms object
# ──────────────────────────────────────────────────────────────────────────────
transformed = supercell.copy()
transformed.set_cell(new_cell)
transformed.set_positions(new_pos)

print("Transformation applied:")
print(f"  Old cell (Å):\n{old_cell.round(4)}")
print(f"  New cell (Å):\n{new_cell.round(4)}")
print(f"  # atoms     : {len(transformed)}")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  Write out
# ──────────────────────────────────────────────────────────────────────────────
#out_path_transformed = cif_path + cif_file_name.replace(
#    '.cif', f'_{rep_a}x{rep_b}x{rep_c}_transformed.xyz'
#)
#ase.io.write(out_path_transformed, transformed, format='extxyz')
#print(f"\nTransformed supercell written to:\n  {out_path_transformed}")

# Repeating and  Cropping the transformed cell into Cuboid (Over Size)

*Compute the axis-aligned bounding box (AABB) of the transformed cell by evaluating all 8 corner combinations of the lattice vectors.\
*Tile the transformed supercell in both positive and negative directions along all three axes to create a dense "atom cloud" that overfills the bounding box on every face.\
*Shift the cloud so the minimum corner sits at the origin, then carve out all atoms that fall within [0, Lx) × [0, Ly) × [0, Lz).\
*Save the result as an orthogonal cuboid XYZ file — this is slightly larger than the target size to ensure no gaps.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1.  Get the transformed cell vectors
# ──────────────────────────────────────────────────────────────────────────────
cell = transformed.get_cell().array
a_vec, b_vec, c_vec = cell[0], cell[1], cell[2]

# ──────────────────────────────────────────────────────────────────────────────
# 2.  Bounding box (same as before)
# ──────────────────────────────────────────────────────────────────────────────
corners = np.array([
    i*a_vec + j*b_vec + k*c_vec
    for i in [0,1] for j in [0,1] for k in [0,1]
])
bbox_min = corners.min(axis=0)
bbox_max = corners.max(axis=0)
bbox     = bbox_max - bbox_min

Lx, Ly, Lz = bbox
print(f"Bounding box (Å): Lx={Lx:.6f}  Ly={Ly:.6f}  Lz={Lz:.6f}")

# ──────────────────────────────────────────────────────────────────────────────
# 3.  Tile in BOTH positive and negative directions along each lattice vector
#     so that after shifting by bbox_min the [0, L) window is fully covered
#     on ALL faces — including the z-face that was ragged before
# ──────────────────────────────────────────────────────────────────────────────
rep_a2 = int(np.ceil(bbox[0] / np.linalg.norm(a_vec))) + 2
rep_b2 = int(np.ceil(bbox[1] / np.linalg.norm(b_vec))) + 2
rep_c2 = int(np.ceil(bbox[2] / np.linalg.norm(c_vec))) + 2

print(f"Tiling (each direction ±): {rep_a2} x {rep_b2} x {rep_c2}")

# Build the tiled positions manually with negative offsets included
base_pos  = transformed.get_positions()   # (M,3)
base_syms = np.array(transformed.get_chemical_symbols())

all_positions = []
all_symbols   = []

for i in range(-rep_a2, rep_a2 + 1):
    for j in range(-rep_b2, rep_b2 + 1):
        for k in range(-rep_c2, rep_c2 + 1):
            offset = i*a_vec + j*b_vec + k*c_vec
            all_positions.append(base_pos + offset)
            all_symbols.append(base_syms)

all_positions = np.vstack(all_positions)
all_symbols   = np.concatenate(all_symbols)

print(f"Total atoms in tiled cloud: {len(all_positions)}")

# ──────────────────────────────────────────────────────────────────────────────
# 4.  Shift so bbox_min → origin, then carve [0, L)
# ──────────────────────────────────────────────────────────────────────────────
pos = all_positions - bbox_min

mask = (
    (pos[:, 0] >= 0) & (pos[:, 0] < Lx) &
    (pos[:, 1] >= 0) & (pos[:, 1] < Ly) &
    (pos[:, 2] >= 0) & (pos[:, 2] < Lz)
)

cuboid = ase.Atoms(
    symbols   = all_symbols[mask],
    positions = pos[mask],
    cell      = [Lx, Ly, Lz],
    pbc       = True,
)

print(f"\nCarved cuboid: {Lx:.4f} x {Ly:.4f} x {Lz:.4f} Å")
print(f"Atoms kept   : {len(cuboid)}  ({cuboid.get_chemical_formula()})")

# Sanity check
unit_cell_vol  = abs(np.dot(a_vec, np.cross(b_vec, c_vec)))
cuboid_vol     = Lx * Ly * Lz
atoms_per_unit = len(transformed) / (rep_a * rep_b * rep_c)
expected_atoms = (cuboid_vol / unit_cell_vol) * atoms_per_unit
print(f"\nExpected atoms (approx): {expected_atoms:.1f}")
print(f"Actual atoms           : {len(cuboid)}")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  Write out
# ──────────────────────────────────────────────────────────────────────────────
#out_path_cuboid = cif_path + cif_file_name.replace(
#    '.cif', f'_{rep_a}x{rep_b}x{rep_c}_cuboid.xyz'
#)
#ase.io.write(out_path_cuboid, cuboid, format='extxyz')
#print(f"\nCuboid written to:\n  {out_path_cuboid}")

# Cropping the transformed cell into Cuboid (Target Size)

*Take the oversized cuboid and crop it symmetrically from the centre to exactly the desired dimensions (e.g., 100 × 100 × 100 Å).\
*Cropping from the centre avoids biasing the structure toward one edge.\
*Shift the remaining atom positions so the final box starts at the origin, then save the result as the final output XYZ file (e.g., LNMO_cropped_100.0x100.0x100.0.xyz).\

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 1.  INPUTS — desired crop dimensions
# ──────────────────────────────────────────────────────────────────────────────
crop_Lx = target_Lx   # Å — desired final x dimension
crop_Ly = target_Ly   # Å — desired final y dimension
crop_Lz = target_Lz   # Å — desired final z dimension

print(f"Input cuboid : {cuboid.get_cell()[0,0]:.4f} x "
      f"{cuboid.get_cell()[1,1]:.4f} x "
      f"{cuboid.get_cell()[2,2]:.4f} Å")
print(f"Crop target  : {crop_Lx:.4f} x {crop_Ly:.4f} x {crop_Lz:.4f} Å")

# ──────────────────────────────────────────────────────────────────────────────
# 2.  Validate — crop must not exceed current cuboid dimensions
# ──────────────────────────────────────────────────────────────────────────────
cur_Lx, cur_Ly, cur_Lz = (cuboid.get_cell()[i, i] for i in range(3))

assert crop_Lx <= cur_Lx + 1e-6, f"crop_Lx={crop_Lx} exceeds cuboid Lx={cur_Lx:.4f}"
assert crop_Ly <= cur_Ly + 1e-6, f"crop_Ly={crop_Ly} exceeds cuboid Ly={cur_Ly:.4f}"
assert crop_Lz <= cur_Lz + 1e-6, f"crop_Lz={crop_Lz} exceeds cuboid Lz={cur_Lz:.4f}"

# ──────────────────────────────────────────────────────────────────────────────
# 3.  Crop from the CENTRE so the structure is not biased to one side
# ──────────────────────────────────────────────────────────────────────────────
pos = cuboid.get_positions()

cx = cur_Lx / 2.0
cy = cur_Ly / 2.0
cz = cur_Lz / 2.0

x_lo, x_hi = cx - crop_Lx / 2.0, cx + crop_Lx / 2.0
y_lo, y_hi = cy - crop_Ly / 2.0, cy + crop_Ly / 2.0
z_lo, z_hi = cz - crop_Lz / 2.0, cz + crop_Lz / 2.0

mask = (
    (pos[:, 0] >= x_lo) & (pos[:, 0] < x_hi) &
    (pos[:, 1] >= y_lo) & (pos[:, 1] < y_hi) &
    (pos[:, 2] >= z_lo) & (pos[:, 2] < z_hi)
)

# ──────────────────────────────────────────────────────────────────────────────
# 4.  Shift positions so the cropped box starts at the origin
# ──────────────────────────────────────────────────────────────────────────────
cropped_pos = pos[mask] - np.array([x_lo, y_lo, z_lo])

symbols_arr = np.array(cuboid.get_chemical_symbols())

cropped = ase.Atoms(
    symbols   = symbols_arr[mask],
    positions = cropped_pos,
    cell      = [crop_Lx, crop_Ly, crop_Lz],
    pbc       = True,
)

print(f"\nAtoms before crop : {len(cuboid)}")
print(f"Atoms after crop  : {len(cropped)}")
print(f"Final cell (Å)    : {crop_Lx:.4f} x {crop_Ly:.4f} x {crop_Lz:.4f}")
print(f"Chemical formula  : {cropped.get_chemical_formula()}")

# ──────────────────────────────────────────────────────────────────────────────
# 5.  Write out
# ──────────────────────────────────────────────────────────────────────────────
out_path_cropped = cif_path + cif_file_name.replace(
    '.cif', f'_cropped_{crop_Lx:.1f}x{crop_Ly:.1f}x{crop_Lz:.1f}.xyz'
)
ase.io.write(out_path_cropped, cropped, format='extxyz')
print(f"\nCropped structure written to:\n  {out_path_cropped}")